# Multi-Fidelity SOFC Dataset Usage Examples

This notebook demonstrates how to load, explore, and use the multi-fidelity SOFC dataset for machine learning.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import sys
import os

# Add parent directory to path
sys.path.append('..')

from src.utils.data_utils import DatasetManager

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Loading the Dataset

In [ ]:
# Initialize dataset manager
manager = DatasetManager('../data')

# Check available datasets
fidelities = ['low_fidelity', 'mid_fidelity', 'high_fidelity', 'experimental']

print("Available datasets:")
for fidelity in fidelities:
    if manager.file_paths[fidelity].exists():
        size_mb = manager.file_paths[fidelity].stat().st_size / 1e6
        print(f"  {fidelity}: {size_mb:.1f} MB")
    else:
        print(f"  {fidelity}: Not found")

In [ ]:
# Load a subset of low-fidelity data
lf_data = manager.read_dataset('low_fidelity', indices=list(range(100)))

print("Low-fidelity data structure:")
print(f"  Input variables: {list(lf_data['inputs'].keys())}")
print(f"  Output variables: {list(lf_data['outputs'].keys())}")
print(f"  Number of samples: {len(lf_data['inputs']['temperature'])}")

## 2. Exploring Input-Output Relationships

In [ ]:
# Create DataFrame for easier analysis
df_lf = pd.DataFrame({
    'temperature': lf_data['inputs']['temperature'],
    'current_density': lf_data['inputs']['current_density'],
    'fuel_utilization': lf_data['inputs']['fuel_utilization'],
    'voltage': lf_data['outputs']['voltage'],
    'power_density': lf_data['outputs']['power_density'],
    'degradation_rate': lf_data['outputs']['degradation_rate']
})

# Display statistics
print("Dataset statistics:")
df_lf.describe()

In [ ]:
# Plot correlations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Temperature vs Voltage
axes[0, 0].scatter(df_lf['temperature'], df_lf['voltage'], alpha=0.5)
axes[0, 0].set_xlabel('Temperature (K)')
axes[0, 0].set_ylabel('Voltage (V)')
axes[0, 0].set_title('Temperature vs Voltage')

# Current vs Voltage (I-V curve)
axes[0, 1].scatter(df_lf['current_density']/10000, df_lf['voltage'], alpha=0.5)
axes[0, 1].set_xlabel('Current Density (A/cm²)')
axes[0, 1].set_ylabel('Voltage (V)')
axes[0, 1].set_title('I-V Characteristic')

# Fuel Utilization vs Power
axes[1, 0].scatter(df_lf['fuel_utilization'], df_lf['power_density'], alpha=0.5)
axes[1, 0].set_xlabel('Fuel Utilization')
axes[1, 0].set_ylabel('Power Density (W/m²)')
axes[1, 0].set_title('Fuel Utilization vs Power')

# Degradation Rate Distribution
axes[1, 1].hist(df_lf['degradation_rate'], bins=30, edgecolor='black')
axes[1, 1].set_xlabel('Degradation Rate (%/1000h)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Degradation Rate Distribution')

plt.tight_layout()
plt.show()

## 3. Multi-Fidelity Data Comparison

In [ ]:
# Load samples from each fidelity level
n_samples = 50
multi_fidelity_data = {}

for fidelity in ['low_fidelity', 'mid_fidelity', 'high_fidelity']:
    try:
        data = manager.read_dataset(fidelity, indices=list(range(min(n_samples, 100))))
        multi_fidelity_data[fidelity] = data
        print(f"Loaded {fidelity}: {len(data['inputs']['temperature'])} samples")
    except:
        print(f"Could not load {fidelity}")

In [ ]:
# Compare voltage predictions across fidelities
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'low_fidelity': 'blue', 'mid_fidelity': 'green', 'high_fidelity': 'red'}

for fidelity, data in multi_fidelity_data.items():
    if 'voltage' in data['outputs']:
        voltages = data['outputs']['voltage']
        ax.hist(voltages, bins=20, alpha=0.5, label=fidelity.replace('_', ' ').title(),
               color=colors[fidelity], density=True)

ax.set_xlabel('Voltage (V)')
ax.set_ylabel('Density')
ax.set_title('Voltage Distribution Across Fidelity Levels')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 4. Spatial Field Analysis (Mid/High Fidelity)

In [ ]:
# Load a single high-fidelity sample with spatial fields
if 'high_fidelity' in multi_fidelity_data:
    hf_data = multi_fidelity_data['high_fidelity']
    
    # Check available spatial fields
    spatial_fields = ['temperature_field', 'current_density_field', 'stress_tensor', 'damage_field']
    
    print("Available spatial fields:")
    for field in spatial_fields:
        if field in hf_data['outputs']:
            shape = hf_data['outputs'][field].shape
            print(f"  {field}: shape = {shape}")

In [ ]:
# Visualize temperature field
if 'high_fidelity' in multi_fidelity_data and 'temperature_field' in hf_data['outputs']:
    T_field = hf_data['outputs']['temperature_field'][0]  # First sample
    
    # Take middle slice if 3D
    if T_field.ndim == 3:
        T_slice = T_field[:, :, T_field.shape[2]//2]
    else:
        T_slice = T_field
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(T_slice, cmap='hot', aspect='auto')
    plt.colorbar(im, ax=ax, label='Temperature (K)')
    ax.set_xlabel('Y coordinate')
    ax.set_ylabel('X coordinate (flow direction)')
    ax.set_title('Temperature Field (Middle Z-Slice)')
    plt.show()
    
    # Plot temperature profile along flow direction
    fig, ax = plt.subplots(figsize=(10, 5))
    T_profile = np.mean(T_slice, axis=1)  # Average across width
    ax.plot(T_profile, linewidth=2)
    ax.set_xlabel('Position along flow direction')
    ax.set_ylabel('Average Temperature (K)')
    ax.set_title('Temperature Rise Along Flow Direction')
    ax.grid(True, alpha=0.3)
    plt.show()

## 5. Preparing Data for Machine Learning

In [ ]:
# Function to prepare ML-ready dataset
def prepare_ml_dataset(data, input_vars, output_vars):
    """Prepare data for machine learning training."""
    
    # Stack input features
    X = []
    for var in input_vars:
        if var in data['inputs']:
            values = data['inputs'][var]
            if values.ndim == 1:
                X.append(values[:, np.newaxis])
            else:
                X.append(values)
    X = np.hstack(X)
    
    # Stack output targets
    y = []
    for var in output_vars:
        if var in data['outputs']:
            values = data['outputs'][var]
            if values.ndim == 1:
                y.append(values[:, np.newaxis])
            else:
                # Flatten spatial fields if needed
                y.append(values.reshape(len(values), -1))
    y = np.hstack(y)
    
    return X, y

# Prepare low-fidelity data
input_vars = ['temperature', 'current_density', 'fuel_utilization', 'pressure']
output_vars = ['voltage', 'power_density', 'degradation_rate']

X_lf, y_lf = prepare_ml_dataset(lf_data, input_vars, output_vars)

print(f"Low-fidelity ML dataset:")
print(f"  X shape: {X_lf.shape}")
print(f"  y shape: {y_lf.shape}")

In [ ]:
# Split into train/test sets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X_lf, y_lf, test_size=0.2, random_state=42)

# Standardize features
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 6. Simple ML Model Example

In [ ]:
# Train a simple neural network
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Create and train model
model = MLPRegressor(hidden_layer_sizes=(50, 30, 20), 
                     activation='relu',
                     max_iter=500,
                     random_state=42)

model.fit(X_train_scaled, y_train_scaled)

# Make predictions
y_pred_scaled = model.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Performance:")
print(f"  MSE: {mse:.6f}")
print(f"  R² Score: {r2:.4f}")

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

output_names = ['Voltage (V)', 'Power Density (W/m²)', 'Degradation Rate (%/1000h)']

for i, name in enumerate(output_names):
    axes[i].scatter(y_test[:, i], y_pred[:, i], alpha=0.5)
    axes[i].plot([y_test[:, i].min(), y_test[:, i].max()], 
                 [y_test[:, i].min(), y_test[:, i].max()], 
                 'r--', lw=2)
    axes[i].set_xlabel(f'True {name}')
    axes[i].set_ylabel(f'Predicted {name}')
    axes[i].set_title(f'{name} Prediction')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Multi-Fidelity Transfer Learning Setup

In [ ]:
# Example setup for multi-fidelity learning
def create_multifidelity_dataset(n_lf=1000, n_mf=100, n_hf=10):
    """Create multi-fidelity training dataset."""
    
    datasets = {}
    
    # Load different amounts from each fidelity
    try:
        lf = manager.read_dataset('low_fidelity', indices=list(range(n_lf)))
        datasets['low'] = prepare_ml_dataset(lf, input_vars, ['voltage'])
    except:
        print("Could not load low-fidelity data")
    
    try:
        mf = manager.read_dataset('mid_fidelity', indices=list(range(n_mf)))
        datasets['mid'] = prepare_ml_dataset(mf, input_vars, ['voltage'])
    except:
        print("Could not load mid-fidelity data")
    
    try:
        hf = manager.read_dataset('high_fidelity', indices=list(range(n_hf)))
        datasets['high'] = prepare_ml_dataset(hf, input_vars, ['voltage'])
    except:
        print("Could not load high-fidelity data")
    
    return datasets

# Create multi-fidelity dataset
mf_datasets = create_multifidelity_dataset()

print("Multi-fidelity dataset sizes:")
for fidelity, (X, y) in mf_datasets.items():
    print(f"  {fidelity}: {X.shape[0]} samples")

## Summary

This notebook demonstrated:
1. Loading multi-fidelity SOFC datasets
2. Exploring input-output relationships
3. Comparing data across fidelity levels
4. Analyzing spatial fields
5. Preparing data for machine learning
6. Training a simple ML model
7. Setting up multi-fidelity transfer learning

Next steps:
- Implement more sophisticated ML models (CNN, GNN, Transformer)
- Apply multi-fidelity learning techniques
- Incorporate physics-informed constraints
- Perform uncertainty quantification